# 15 · Bibliographic coupling on the BOUN side (Experiment B, descriptive)

**Question.** On the app's own data, how often does a new paper from a followed institution share references with Boğaziçi (BOUN) works, and do coupling and the embedding pick the same researchers? There are no relevance labels here, so **nothing in this notebook is an accuracy claim**. Notebook 14 holds the labelled test.

**Choices fixed before looking at results.**
- **Sample.** The 200 most recent articles (15–19 Oct 2026) from the 7 followed institutions with an abstract and no BOUN author (`scripts/experiment_b_data.py sample`, 2 OpenAlex list calls).
- **Index.** The 7,822 cleaned BOUN works (`data/boun/`), title + abstract (OpenAlex, else the gitignored Crossref/S2 cache), all at full weight (large collaborations included).
- **Coupling.** IDF-weighted cosine over `referenced_works`, researcher score = max over their works (as in notebook 14). IDF is local document frequency over the BOUN works with references plus the sample: global counts for ~238k referenced works would cost about 4,800 list calls, and this notebook makes no accuracy claim.
- **Non-trivial reference:** cited by at most 1% of the BOUN works that have references.
- **Embedding.** Qwen3-Embedding-8B (same server and setup as notebook 14), cosine, researcher score = max over their works.
- **Agreement.** For each sample paper with any coupling signal: how many of coupling's top-10 researchers (those with a score > 0, at most 10) are in the embedding's top-10.
- **Sanity check** (brief: "311 works"; that count came from the old `boun.csv`). In the cleaned data, 642 BOUN works have a co-author from a followed institution; 460 are 100+-author collaborations, so the check uses the other 182, minus those without references. Each is a query against an index of BOUN works published strictly before it (year-only dates count as 31 Dec for index works and 1 Jan for the query), without the query and its duplicate ids. Target = the query's own BOUN authors. **Confound:** co-authors share self-citations, which favours coupling; the embedding has the analogous advantage of shared topic vocabulary.

In [1]:
import gzip
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

from src import bibcoupling as bc
from src.boun_data import work_texts
from src.rankers_v2 import EmbeddingCache

FOLLOWED = {"I97018004", "I136199984", "I95457486", "I1291425158", "I4210090411", "I27837315", "I63966007"}
OUT = RESEARCH / "v2" / "goldstandard"

## Data

In [2]:
works = pd.read_csv(RESEARCH / "data" / "boun" / "boun_works.csv.gz", keep_default_na=False)
works["duplicate_ids"] = works["duplicate_ids"].map(json.loads)
links = pd.read_csv(RESEARCH / "data" / "boun" / "researcher_works.csv.gz")
by_researcher = links.groupby("researcher_id")["work_id"].apply(list).to_dict()
authors_of = links.groupby("work_id")["researcher_id"].apply(set).to_dict()

refs, followed_coauthor = {}, set()
ids = set(works.id)
with gzip.open(RESEARCH / "data" / "boun" / "works_openalex.jsonl.gz", "rt") as f:
    for line in f:
        w = json.loads(line)
        if w["id"] not in ids:
            continue
        refs[w["id"]] = set(w.get("referenced_works") or [])
        lineage = {i.rsplit("/", 1)[-1].upper() for a in w.get("authorships") or []
                   for inst in a.get("institutions") or [] for i in inst.get("lineage") or []}
        if lineage & FOLLOWED:
            followed_coauthor.add(w["id"])

sample = [json.loads(l) for l in open(RESEARCH / "data" / "external" / "followed_sample" / "works.jsonl")]
for s in sample:
    s["refs"] = set(s["referenced_works"])

texts = work_texts(RESEARCH)
print(f"BOUN works {len(works):,}, with references {sum(map(bool, refs.values())):,}; researchers {len(by_researcher):,}")
print(f"sample {len(sample)} ({min(s['publication_date'] for s in sample)} to {max(s['publication_date'] for s in sample)}), "
      f"with references {sum(bool(s['refs']) for s in sample)}")

BOUN works 7,822, with references 6,401; researchers 3,648
sample 200 (2026-10-15 to 2026-10-19), with references 101


In [3]:
df = Counter(r for rs in refs.values() for r in rs)
n_boun_refs = sum(map(bool, refs.values()))
TRIVIAL = 0.01 * n_boun_refs
df_all = df + Counter(r for s in sample for r in s["refs"])
idf = bc.idf_weights(df_all, n_boun_refs + sum(bool(s["refs"]) for s in sample))
print(f"distinct referenced works in BOUN {len(df):,}; trivial (cited by > {TRIVIAL:.0f} BOUN works) {sum(c > TRIVIAL for c in df.values())}")

cache = EmbeddingCache("qwen", RESEARCH / "data" / "v2" / "embeddings_boun")
work_ids = list(works.id)
W = cache.get([texts[i] for i in work_ids])
W = W / np.linalg.norm(W, axis=1, keepdims=True)
row = {w: i for i, w in enumerate(work_ids)}

distinct referenced works in BOUN 236,122; trivial (cited by > 64 BOUN works) 135


## Coverage: shared references between new papers and BOUN works

In [4]:
inv = defaultdict(set)
for w, rs in refs.items():
    for r in rs:
        inv[r].add(w)

cov = []
for s in sample:
    shared = {w for r in s["refs"] for w in inv.get(r, ())}
    nontriv = {w for r in s["refs"] if df.get(r, 0) <= TRIVIAL for w in inv.get(r, ())}
    cov.append({"id": s["id"], "has_refs": bool(s["refs"]), "boun_works_any": len(shared), "boun_works_nontrivial": len(nontriv),
                "researchers_nontrivial": len(set().union(*(authors_of.get(w, set()) for w in nontriv))) if nontriv else 0})
cov = pd.DataFrame(cov)
with_refs = cov[cov.has_refs]
summary = pd.Series({
    "sample papers": len(cov),
    "with references": len(with_refs),
    "share ≥1 reference with a BOUN work": int((with_refs.boun_works_any > 0).sum()),
    "share ≥1 non-trivial reference": int((with_refs.boun_works_nontrivial > 0).sum()),
    "median BOUN researchers reached (non-trivial, among those with any)": float(with_refs.loc[with_refs.boun_works_nontrivial > 0, "researchers_nontrivial"].median()),
})
summary

sample papers                                                          200.0
with references                                                        101.0
share ≥1 reference with a BOUN work                                     52.0
share ≥1 non-trivial reference                                          52.0
median BOUN researchers reached (non-trivial, among those with any)      6.0
dtype: float64

## Agreement between coupling and embedding top-10 researchers

In [5]:
def researcher_scores(work_scores: dict) -> dict:
    out = {}
    for w, s in work_scores.items():
        for r in authors_of.get(w, ()):
            if s > out.get(r, -np.inf):
                out[r] = s
    return out

def top(scores: dict, k=10, positive=False):
    items = [(r, s) for r, s in scores.items() if s > 0 or not positive]
    return [r for r, _ in sorted(items, key=lambda x: (-x[1], x[0]))[:k]]

agree = []
for s in sample:
    if not s["refs"]:
        continue
    cand = {w for r in s["refs"] for w in inv.get(r, ())}
    if not cand:
        continue
    c = researcher_scores({w: bc.coupling(s["refs"], refs[w], "idf_cosine", idf) for w in cand})
    q = cache.get([f"{s['title']} {s['abstract']}".strip()])[0]
    e = researcher_scores(dict(zip(work_ids, W @ (q / np.linalg.norm(q)))))
    tc, te = top(c, positive=True), top(e)
    agree.append({"id": s["id"], "title": s["title"][:70], "coupling researchers": len(tc),
                  "in embedding top-10": len(set(tc) & set(te)),
                  "coupling top-1 rank in embedding": sorted(e, key=lambda r: (-e[r], r)).index(tc[0]) + 1})
agree = pd.DataFrame(agree)
print(f"papers with any coupling signal: {len(agree)}")
print(f"of coupling's top researchers (≤10), share also in the embedding top-10: "
      f"{agree['in embedding top-10'].sum() / agree['coupling researchers'].sum():.1%}")
print("coupling top-1 researcher's rank in the embedding ranking: median",
      agree["coupling top-1 rank in embedding"].median(), "of", len(by_researcher))
agree.describe().round(1)

papers with any coupling signal: 52
of coupling's top researchers (≤10), share also in the embedding top-10: 22.9%
coupling top-1 researcher's rank in the embedding ranking: median 17.0 of 3648


,coupling researchers,in embedding top-10,coupling top-1 rank in embedding
count,52.0,52.0,52.0
mean,6.1,1.4,151.7
std,3.7,1.6,315.7
min,1.0,0.0,1.0
25%,2.0,0.0,5.0
50%,6.0,1.0,17.0
75%,10.0,2.0,88.2
max,10.0,7.0,1435.0


## Sanity check: BOUN works with a followed-institution co-author

In [6]:
def eff(date, year_only, as_index):
    return (date[:4] + "-12-31") if (year_only and as_index) else date

w_info = works.set_index("id")
queries = [w for w in followed_coauthor
           if str(w_info.at[w, "large_collaboration"]) != "True" and refs.get(w)]
print(f"followed-co-author works {len(followed_coauthor)}, not 100+-author {sum(str(w_info.at[w, 'large_collaboration']) != 'True' for w in followed_coauthor)}, with references {len(queries)}")

index_date = {r.id: eff(r.publication_date, str(r.year_only_date) == "True", True) for r in works.itertuples()}
rows = []
for qid in sorted(queries):
    qdate = w_info.at[qid, "publication_date"]
    drop = {qid, *w_info.at[qid, "duplicate_ids"]}
    allowed = [w for w in work_ids if index_date[w] < qdate and w not in drop]
    allowed_set = set(allowed)
    target = authors_of.get(qid, set())
    findable = {r for r in target if any(w in allowed_set for w in by_researcher.get(r, ()))}
    if not findable:
        continue
    cand = {w for r in refs[qid] for w in inv.get(r, ()) if w in allowed_set}
    c = researcher_scores({w: bc.coupling(refs[qid], refs[w], "idf_cosine", idf) for w in cand})
    idx = np.array([row[w] for w in allowed])
    e = researcher_scores(dict(zip(allowed, W[idx] @ W[row[qid]])))
    rows.append({"query": qid, "findable authors": len(findable),
                 "coupling hit@10": bool(set(top(c, positive=True)) & findable),
                 "embedding hit@10": bool(set(top(e)) & findable),
                 "coupling has signal": bool(c)})
sanity = pd.DataFrame(rows)
print(f"queries with ≥1 findable BOUN author: {len(sanity)}")
sanity[["coupling has signal", "coupling hit@10", "embedding hit@10"]].mean().round(3)

followed-co-author works 642, not 100+-author 182, with references 152


queries with ≥1 findable BOUN author: 110


coupling has signal    0.927
coupling hit@10        0.727
embedding hit@10       0.800
dtype: float64

In [7]:
both = pd.crosstab(sanity["coupling hit@10"], sanity["embedding hit@10"])
print("rows: coupling hit@10, columns: embedding hit@10")
both

rows: coupling hit@10, columns: embedding hit@10


embedding hit@10,False,True
coupling hit@10,,
False,16,14
True,6,74


In [8]:
pd.Series({**summary.to_dict(),
           "papers with coupling signal": len(agree),
           "coupling top researchers also in embedding top-10 (share)": agree["in embedding top-10"].sum() / agree["coupling researchers"].sum(),
           "sanity queries": len(sanity),
           "sanity coupling hit@10": sanity["coupling hit@10"].mean(),
           "sanity embedding hit@10": sanity["embedding hit@10"].mean()}).to_csv(OUT / "15_boun_summary.csv", header=["value"])
print("written", OUT / "15_boun_summary.csv")

written /Users/senaoz/Developer/SWE-599/research/v2/goldstandard/15_boun_summary.csv
